In [43]:
import pandas as pd
import matplotlib.pyplot as plt

path = "../datasets/greengrid_v2_2/raw_data/"

In [11]:
def cat_clean(data: pd.DataFrame):
    """Cleaning inconsistency in categorical data"""
    columns = list(data.columns)
    for column in columns:
        if data[column].dtypes == "object":
           data[column] = [i.upper().strip() for i in data[column].astype(str)]

In [30]:
def cat_flag(data: pd.DataFrame):
    """Flagging inconsistency in categorical data"""
    columns = list(data.columns)
    for column in columns:
        if data[column].dtypes == "object":
            a = list(
                    set(
                        [i.upper().strip() for i in list(data[column].unique().astype(str))]
                        )
                )
            b = list(data[column].unique())
            if len(a) == len(b):
                print(column, "->", "Consistent")
            else:
                print(column, "->", "Inconsistent")

## Asset

In [31]:
assets = path + "asset_registry_raw.csv"

df_assets = pd.read_csv(assets)

df_assets

,asset_id,asset_name,technology,region,grid_node,capacity_mw,commission_date,asset_status,source_system
0,SOL-N01,North Solar One,Solar PV,North,NORTH_A,180,2023-06-01,Operational,ASSET_REGISTRY
1,SOL-N02,North Solar Two,Solar PV,North,NORTH_A,140,2024-03-15,Operational,ASSET_REGISTRY
2,WND-N01,North Wind One,WIND,North,NORTH_B,220,2022-10-01,Operational,ASSET_REGISTRY
3,SOL-C01,Central Solar One,Solar PV,Central,CENTRAL_A,160,2023-01-10,Operational,ASSET_REGISTRY
4,SOL-C02,Central Solar Two,Solar PV,Central,CENTRAL_A,110,2024-07-01,Operational,ASSET_REGISTRY
5,WND-C01,Central Wind One,wind,central,CENTRAL_B,180,2022-05-20,Operational,ASSET_REGISTRY


In [32]:
cat_clean(df_assets)
cat_flag(df_assets)

asset_id -> Consistent
asset_name -> Consistent
technology -> Consistent
region -> Consistent
grid_node -> Consistent
commission_date -> Consistent
asset_status -> Consistent
source_system -> Consistent


In [33]:
df_assets = df_assets.drop_duplicates()
df_assets.duplicated().sum()

np.int64(0)

In [34]:
df_assets = df_assets.dropna()
df_assets.isnull().sum()

asset_id           0
asset_name         0
technology         0
region             0
grid_node          0
capacity_mw        0
commission_date    0
asset_status       0
source_system      0
dtype: int64

## BESS

In [63]:
bess = path + "bess_operations_raw.csv"

df_bess = pd.read_csv(bess)

df_bess.head()

,timestamp,region,battery_power_capacity_mw,battery_energy_capacity_mwh,charge_mwh,discharge_mwh,source_system,record_status
0,2025-01-01 00:00:00,North,50,200,0.0,0.0,BESS_CONTROLLER,VALID
1,2025-01-01 00:00:00,Central,50,200,0.0,0.0,BESS_CONTROLLER,VALID
2,2025-01-01 01:00:00,North,50,200,0.0,0.0,BESS_CONTROLLER,VALID
3,2025-01-01 01:00:00,Central,50,200,0.0,0.0,BESS_CONTROLLER,VALID
4,2025-01-01 02:00:00,North,50,200,0.0,0.0,BESS_CONTROLLER,VALID


In [64]:
cat_clean(df_bess)
cat_flag(df_bess)

timestamp -> Consistent
region -> Consistent
source_system -> Consistent
record_status -> Consistent


In [65]:
df_bess = df_bess.drop_duplicates()
df_bess.duplicated().sum()

np.int64(0)

In [66]:
df_bess = df_bess.dropna()
df_bess.isnull().sum()

timestamp                      0
region                         0
battery_power_capacity_mw      0
battery_energy_capacity_mwh    0
charge_mwh                     0
discharge_mwh                  0
source_system                  0
record_status                  0
dtype: int64

In [67]:
df_bess.describe()

,battery_power_capacity_mw,battery_energy_capacity_mwh,charge_mwh,discharge_mwh
count,17469.0,17469.0,17469.000000,17469.000000
mean,50.0,200.0,2.765242,2.212873
std,0.0,0.0,18.105486,8.571690
min,50.0,200.0,-10.000000,0.000000
25%,50.0,200.0,0.000000,0.000000
50%,50.0,200.0,0.000000,0.000000
75%,50.0,200.0,0.000000,0.000000
max,50.0,200.0,999.000000,37.500000


In [68]:
df_bess[df_bess["charge_mwh"] >= 0][df_bess["charge_mwh"] != 999].describe()

C:\Users\User\AppData\Local\Temp\ipykernel_2184\2627907241.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_bess[df_bess["charge_mwh"] >= 0][df_bess["charge_mwh"] != 999].describe()


,battery_power_capacity_mw,battery_energy_capacity_mwh,charge_mwh,discharge_mwh
count,17461.0,17461.0,17461.000000,17461.000000
mean,50.0,200.0,2.539947,2.213887
std,0.0,0.0,10.024850,8.573523
min,50.0,200.0,0.000000,0.000000
25%,50.0,200.0,0.000000,0.000000
50%,50.0,200.0,0.000000,0.000000
75%,50.0,200.0,0.000000,0.000000
max,50.0,200.0,50.000000,37.500000


Menggunakan metode **Capping**, di mana nilai negatif diganti menjadi 0 dan 999 diganti menjadi 50

In [69]:
df_bess["charge_mwh"][df_bess["charge_mwh"] < 0] = 0
df_bess["charge_mwh"][df_bess["charge_mwh"] == 999] = 50


C:\Users\User\AppData\Local\Temp\ipykernel_2184\3150787036.py:1: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_bess["charge_mwh"][df_bess["charge_mwh"] < 0] = 0
C:\Users\User\AppData\Local\Temp\ipykernel_2184\3150787036.py:2: FutureWarnin

In [70]:
df_bess.describe()

,battery_power_capacity_mw,battery_energy_capacity_mwh,charge_mwh,discharge_mwh
count,17469.0,17469.0,17469.000000,17469.000000
mean,50.0,200.0,2.550232,2.212873
std,0.0,0.0,10.048321,8.571690
min,50.0,200.0,0.000000,0.000000
25%,50.0,200.0,0.000000,0.000000
50%,50.0,200.0,0.000000,0.000000
75%,50.0,200.0,0.000000,0.000000
max,50.0,200.0,50.000000,37.500000


## Dispatch